In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

# ══════════════════════════════════════════════════════
#  NAIVE BAYES CLASSIFIER — HÀM TÁI SỬ DỤNG
# ══════════════════════════════════════════════════════

class NaiveBayesDA:
    """
    Naive Bayes cho Categorical Data + Laplace Smoothing.
    Tái sử dụng cho MỌI bài toán phân loại categorical.
    """

    def fit(self, X, y, feature_names):
        self.classes       = list(set(y))
        self.feature_names = feature_names
        n                  = len(y)

        self.prior = {
            c: sum(1 for yi in y if yi == c) / n
            for c in self.classes
        }

        self.cond = defaultdict(lambda: defaultdict(dict))
        for fi, fname in enumerate(feature_names):
            vals = list(set(X[:, fi]))
            for cls in self.classes:
                cls_count = sum(1 for yi in y if yi == cls)
                for v in vals:
                    cnt = sum(
                        1 for xi, yi in zip(X[:, fi], y)
                        if yi == cls and xi == v
                    )
                    self.cond[fname][v][cls] = (
                        (cnt + 1) / (cls_count + len(vals))
                    )
        return self

    def predict(self, x_input, verbose=True):
        scores = {}

        if verbose:
            print(f"\nInput: {dict(zip(self.feature_names, x_input))}")
            print("─" * 55)

        for cls in self.classes:
            score = self.prior[cls]
            if verbose:
                print(f"\n  Class='{cls}' | Prior={self.prior[cls]:.3f}")

            for fname, val in zip(self.feature_names, x_input):
                p = self.cond[fname].get(val, {}).get(cls, 0.001)
                score *= p
                if verbose:
                    print(f"    × P({fname}={val}|{cls}) = {p:.3f}")

            scores[cls] = score
            if verbose:
                print(f"    → Score = {score:.6f}")

        total = sum(scores.values())
        probs = {c: s / total for c, s in scores.items()}
        best  = max(probs, key=probs.get)

        if verbose:
            print(f"\n{'─'*55}")
            print("  POSTERIOR:")
            for c, p in sorted(probs.items(), key=lambda x: -x[1]):
                bar = "█" * int(p * 35)
                print(f"  {c}: {p:.1%}  {bar}")

        return best, probs


# ══════════════════════════════════════════════════════
#  DATASET 1: TENNIS (Đã train ở phần giải tay)
# ══════════════════════════════════════════════════════

tennis_data = [
    ['Sunny',    'Hot',  'High',   'Weak',   'No'],
    ['Sunny',    'Hot',  'High',   'Strong', 'No'],
    ['Overcast', 'Hot',  'High',   'Weak',   'Yes'],
    ['Rainy',    'Mild', 'High',   'Weak',   'Yes'],
    ['Rainy',    'Cool', 'Normal', 'Weak',   'Yes'],
    ['Rainy',    'Cool', 'Normal', 'Strong', 'No'],
    ['Overcast', 'Cool', 'Normal', 'Strong', 'Yes'],
    ['Sunny',    'Mild', 'High',   'Weak',   'No'],
    ['Sunny',    'Cool', 'Normal', 'Weak',   'Yes'],
    ['Rainy',    'Mild', 'Normal', 'Weak',   'Yes'],
]
TENNIS_FEATURES = ['Outlook', 'Temp', 'Humidity', 'Wind']
arr_tennis = np.array(tennis_data)

model_tennis = NaiveBayesDA().fit(
    arr_tennis[:, :-1], arr_tennis[:, -1], TENNIS_FEATURES
)

print("=" * 60)
print("  DATASET 1: TENNIS — Xác nhận kết quả giải tay")
print("=" * 60)
pred_tennis, probs_tennis = model_tennis.predict(
    ['Sunny', 'Cool', 'High', 'Strong']
)
print(f"\n  → Kết luận: {pred_tennis} "
      f"(P={probs_tennis[pred_tennis]:.1%})")


# ══════════════════════════════════════════════════════
#  DATASET 2: PHÊ DUYỆT VAY VỐN (Loan Approval)
#  Chứng minh hàm NaiveBayesDA() DÙNG LẠI được 100%
#  Chỉ cần đổi: data, features, input cần predict
# ══════════════════════════════════════════════════════

print("\n\n" + "=" * 60)
print("  DATASET 2: PHÊ DUYỆT VAY VỐN NGÂN HÀNG")
print("  (Thử hàm tái sử dụng với bài toán hoàn toàn khác)")
print("=" * 60)

loan_data = [
    # [Income,  CreditHistory, HasAsset, Approved]
    ['High', 'Good', 'Yes', 'Approved'],
    ['Low',  'Bad',  'No',  'Rejected'],
    ['High', 'Bad',  'No',  'Rejected'],
    ['Mid',  'Good', 'No',  'Approved'],
    ['Mid',  'Good', 'Yes', 'Approved'],
    ['Low',  'Good', 'No',  'Rejected'],
    ['High', 'Good', 'No',  'Approved'],
    ['Low',  'Bad',  'Yes', 'Rejected'],
    ['Mid',  'Bad',  'No',  'Rejected'],
    ['High', 'Good', 'Yes', 'Approved'],
    ['Low',  'Good', 'Yes', 'Approved'],
    ['Mid',  'Bad',  'Yes', 'Rejected'],
]
LOAN_FEATURES = ['Income', 'CreditHistory', 'HasAsset']
arr_loan = np.array(loan_data)

print("\n  DATASET:")
df_loan_display = pd.DataFrame(
    loan_data, columns=LOAN_FEATURES + ['Approved']
)
print(df_loan_display.to_string(index=False))

model_loan = NaiveBayesDA().fit(
    arr_loan[:, :-1], arr_loan[:, -1], LOAN_FEATURES
)

print("\n" + "─" * 60)
print("  TEST CASE: Khách hàng thu nhập thấp,")
print("  lịch sử tín dụng tốt, có tài sản thế chấp")
print("─" * 60)

new_customer = ['Low', 'Good', 'Yes']
pred_loan, probs_loan = model_loan.predict(new_customer)

print(f"\n  📋 KẾT LUẬN CHO VAY:")
print(f"  Quyết định: {pred_loan}")
print(f"  Độ tin cậy: {probs_loan[pred_loan]:.1%}")

if pred_loan == 'Approved':
    print(f"  ✅ NÊN phê duyệt khoản vay")
else:
    print(f"  ❌ KHÔNG nên phê duyệt khoản vay")


# ── TEST THÊM NHIỀU TRƯỜNG HỢP (Batch testing) ───────
print("\n" + "─" * 60)
print("  TEST NHIỀU KHÁCH HÀNG CÙNG LÚC")
print("─" * 60)

test_customers = [
    (['High', 'Good', 'Yes'], "KH A: Thu nhập cao, tốt mọi mặt"),
    (['Low',  'Bad',  'No'],  "KH B: Thu nhập thấp, xấu mọi mặt"),
    (['Mid',  'Good', 'No'],  "KH C: Trung bình, không tài sản"),
    (['High', 'Bad',  'Yes'], "KH D: Thu nhập cao nhưng lịch sử xấu"),
]

print(f"\n  {'Mô tả':<40} {'Dự đoán':>12} {'P(Approved)':>12}")
print("  " + "─" * 66)
for customer, desc in test_customers:
    pred, probs = model_loan.predict(customer, verbose=False)
    p_approved = probs.get('Approved', 0)
    icon = "✅" if pred == "Approved" else "❌"
    print(f"  {desc:<40} {icon} {pred:<9} {p_approved:>11.1%}")

print(f"\n  💡 INSIGHT:")
print(f"  → Hàm NaiveBayesDA() không cần sửa code")
print(f"  → Chỉ thay data + feature_names + input")
print(f"  → Dùng được cho: Tennis, Loan, Spam, Churn, ...")

  DATASET 1: TENNIS — Xác nhận kết quả giải tay

Input: {'Outlook': 'Sunny', 'Temp': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
───────────────────────────────────────────────────────

  Class='Yes' | Prior=0.600
    × P(Outlook=Sunny|Yes) = 0.222
    × P(Temp=Cool|Yes) = 0.444
    × P(Humidity=High|Yes) = 0.375
    × P(Wind=Strong|Yes) = 0.250
    → Score = 0.005556

  Class='No' | Prior=0.400
    × P(Outlook=Sunny|No) = 0.571
    × P(Temp=Cool|No) = 0.286
    × P(Humidity=High|No) = 0.667
    × P(Wind=Strong|No) = 0.500
    → Score = 0.021769

───────────────────────────────────────────────────────
  POSTERIOR:
  No: 79.7%  ███████████████████████████
  Yes: 20.3%  ███████

  → Kết luận: No (P=79.7%)


  DATASET 2: PHÊ DUYỆT VAY VỐN NGÂN HÀNG
  (Thử hàm tái sử dụng với bài toán hoàn toàn khác)

  DATASET:
Income CreditHistory HasAsset Approved
  High          Good      Yes Approved
   Low           Bad       No Rejected
  High           Bad       No Rejected
   Mid          Good  

In [3]:
# ============================================================
# CACH DUNG THONG NHAT: dataset nao cung theo cung mot mau
# ============================================================

def run_naive_bayes(name, data, feature_names, test_input):
    data_array = np.array(data)

    if data_array.ndim != 2 or data_array.shape[1] != len(feature_names) + 1:
        raise ValueError("data phai gom cac cot feature va 1 cot nhan o cuoi")
    if len(test_input) != len(feature_names):
        raise ValueError("test_input phai co dung so feature")

    model = NaiveBayesDA().fit(
        data_array[:, :-1], data_array[:, -1], feature_names
    )
    prediction, probabilities = model.predict(test_input, verbose=False)

    print(f"\n{name}")
    print(f"Input: {dict(zip(feature_names, test_input))}")
    print(f"Du doan: {prediction}")
    print(f"Do tin cay: {probabilities[prediction]:.1%}")
    return model, prediction, probabilities

model_tennis, pred_tennis, probs_tennis = run_naive_bayes(
    "TENNIS",
    tennis_data,
    TENNIS_FEATURES,
    ['Sunny', 'Cool', 'High', 'Strong'],
)

model_loan, pred_loan, probs_loan = run_naive_bayes(
    "LOAN APPROVAL",
    loan_data,
    LOAN_FEATURES,
    ['Low', 'Good', 'Yes'],
)


TENNIS
Input: {'Outlook': 'Sunny', 'Temp': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Du doan: No
Do tin cay: 79.7%

LOAN APPROVAL
Input: {'Income': 'Low', 'CreditHistory': 'Good', 'HasAsset': 'Yes'}
Du doan: Approved
Do tin cay: 74.5%


In [5]:
# ============================================================
# DATASET 3: DU BAO KHACH HANG ROI BO DICH VU (5 feature)
# ============================================================

churn_data = [
    # [Plan, Usage, Tickets, Contract, AutoPay, Churn]
    ['Basic',   'Low',  'Many', 'Monthly', 'No',  'Yes'],
    ['Basic',   'Low',  'Few',  'Monthly', 'No',  'Yes'],
    ['Basic',   'High', 'Few',  'Monthly', 'Yes', 'No'],
    ['Basic',   'High', 'Few',  'Annual',  'Yes', 'No'],
    ['Premium', 'High', 'Few',  'Annual',  'Yes', 'No'],
    ['Premium', 'High', 'Few',  'Monthly', 'Yes', 'No'],
    ['Premium', 'Low',  'Many', 'Monthly', 'No',  'Yes'],
    ['Premium', 'Low',  'Few',  'Annual',  'Yes', 'No'],
    ['Basic',   'Low',  'Many', 'Annual',  'No',  'Yes'],
    ['Premium', 'High', 'Many', 'Monthly', 'No',  'Yes'],
    ['Basic',   'High', 'Many', 'Monthly', 'Yes', 'Yes'],
    ['Premium', 'Low',  'Few',  'Monthly', 'Yes', 'No'],
]
CHURN_FEATURES = ['Plan', 'Usage', 'Tickets', 'Contract', 'AutoPay']

model_churn, pred_churn, probs_churn = run_naive_bayes(
    "CUSTOMER CHURN",
    churn_data,
    CHURN_FEATURES,
    ['Basic', 'Low', 'Many', 'Monthly', 'Yes'],
)


CUSTOMER CHURN
Input: {'Plan': 'Basic', 'Usage': 'Low', 'Tickets': 'Many', 'Contract': 'Monthly', 'AutoPay': 'Yes'}
Du doan: Yes
Do tin cay: 87.7%


In [6]:
df = pd.DataFrame(churn_data, columns=CHURN_FEATURES + ['Status'])

In [7]:
model_churn, pred_churn, probs_churn = run_naive_bayes(
    "CUSTOMER CHURN",
    df,
    CHURN_FEATURES,
    ['Basic', 'Low', 'Many', 'Monthly', 'Yes'],
)


CUSTOMER CHURN
Input: {'Plan': 'Basic', 'Usage': 'Low', 'Tickets': 'Many', 'Contract': 'Monthly', 'AutoPay': 'Yes'}
Du doan: Yes
Do tin cay: 87.7%


In [5]:
tennis_data = [
    ['Sunny',    'Hot',  'High',   'Weak',   'No'],
    ['Sunny',    'Hot',  'High',   'Strong', 'No'],
    ['Overcast', 'Hot',  'High',   'Weak',   'Yes'],
    ['Rainy',    'Mild', 'High',   'Weak',   'Yes'],
    ['Rainy',    'Cool', 'Normal', 'Weak',   'Yes'],
    ['Rainy',    'Cool', 'Normal', 'Strong', 'No'],
    ['Overcast', 'Cool', 'Normal', 'Strong', 'Yes'],
    ['Sunny',    'Mild', 'High',   'Weak',   'No'],
    ['Sunny',    'Cool', 'Normal', 'Weak',   'Yes'],
    ['Rainy',    'Mild', 'Normal', 'Weak',   'Yes'],
]

In [7]:
from Naive_Bayes_Classifier import NaiveBayesDA
import numpy as np

model = NaiveBayesDA()

In [8]:
data = np.array(tennis_data)

X = data[:, :-1]
y = data[:, -1]

feature_names = [
    "Outlook",
    "Temperature",
    "Humidity",
    "Wind"
]


In [9]:
model.fit(
    X,
    y,
    feature_names
)

In [10]:
test_input = [
    'Sunny',
    'Cool',
    'High',
    'Strong'
]

prediction, probabilities = model.predict(
    test_input,
    verbose=True
)


# =========================
# 4. PREDICT NHANH
# =========================




Input: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
───────────────────────────────────────────────────────

  Class='No' | Prior=0.400
    × P(Outlook=Sunny|No) = 0.571
    × P(Temperature=Cool|No) = 0.286
    × P(Humidity=High|No) = 0.667
    × P(Wind=Strong|No) = 0.500
    → Score = 0.021769

  Class='Yes' | Prior=0.600
    × P(Outlook=Sunny|Yes) = 0.222
    × P(Temperature=Cool|Yes) = 0.444
    × P(Humidity=High|Yes) = 0.375
    × P(Wind=Strong|Yes) = 0.250
    → Score = 0.005556

───────────────────────────────────────────────────────
  POSTERIOR:
  No: 79.7%  ███████████████████████████
  Yes: 20.3%  ███████


In [11]:
model.predict_fast(
    test_input,
    name="Tennis Test"
)


[Tennis Test]
Input: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Du doan: No
Do tin cay: 79.7%


(np.str_('No'),
 {np.str_('No'): 0.7966804979253111, np.str_('Yes'): 0.2033195020746888})

In [14]:
import pandas as pd
tennis_df = pd.DataFrame(tennis_data, columns=feature_names + ['Status'])
feature_names = [
    'Outlook',
    'Temperature',
    'Humidity',
    'Wind'
]

X = tennis_df[feature_names].values
y = tennis_df['Status'].values